# CachePredict

Predicts the next API call from a session history.
Run the cells in sequence (Shift+Enter through all of them).

In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder


In [2]:
df = pd.read_csv("synthetic_api_calls.csv")
df["url_params"] = df["url_params"].fillna("")
df["body"] = df["body"].fillna("")

# every unique call string becomes an integer (the vocab)
all_calls = pd.concat([df["current_call"], df["next_call"]]).unique()
le = LabelEncoder().fit(all_calls)
vocab_size = len(le.classes_)
print(f"vocab size: {vocab_size}")

df["x_id"] = le.transform(df["current_call"])
df["y_id"] = le.transform(df["next_call"])
df[["session_id", "step", "current_call", "next_call"]].head()


vocab size: 142


,session_id,step,current_call,next_call
0,1,1,GET /products/filter,"POST /products/filter body={""gender"":""female""..."
1,1,2,"POST /products/filter body={""gender"":""female""...",GET /products?id=47
2,1,3,GET /products?id=47,GET /cart?product_id=47
3,1,4,GET /cart?product_id=47,END
4,2,1,GET /products,GET /products?id=88


In [5]:
df["x_id"]

0       23
1       68
2       35
3       12
4       22
        ..
7801     8
7802    22
7803    24
7804    35
7805    12
Name: x_id, Length: 7806, dtype: int32

In [6]:
df

,session_id,step,current_call,next_call,endpoint,method,url_params,body,user_segment,device,hour,is_last_step,x_id,y_id
0,1,1,GET /products/filter,"POST /products/filter body={""gender"":""female""...",/products/filter,GET,,,browser,mobile,23,False,23,68
1,1,2,"POST /products/filter body={""gender"":""female""...",GET /products?id=47,/products/filter,POST,,"{""gender"":""female"",""category"":""dress"",""price"":...",browser,mobile,23,False,68,35
2,1,3,GET /products?id=47,GET /cart?product_id=47,/products,GET,id=47,,browser,mobile,23,False,35,12
3,1,4,GET /cart?product_id=47,END,/cart,GET,product_id=47,,browser,mobile,23,True,12,0
4,2,1,GET /products,GET /products?id=88,/products,GET,,,new_visitor,desktop,1,False,22,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7801,1999,3,GET /cart?product_id=33,END,/cart,GET,product_id=33,,new_visitor,tablet,14,True,8,0
7802,2000,1,GET /products,GET /products?id=101,/products,GET,,,buyer,tablet,23,False,22,24
7803,2000,2,GET /products?id=101,GET /products?id=47,/products,GET,id=101,,buyer,tablet,23,False,24,35
7804,2000,3,GET /products?id=47,GET /cart?product_id=47,/products,GET,id=47,,buyer,tablet,23,False,35,12


In [7]:
# turn each session into (prefix -> next) pairs
# session [A, B, C] becomes  ([A] -> B), ([A,B] -> C)
sessions = (
    df.sort_values(["session_id", "step"])
    .groupby("session_id")["x_id"]
    .apply(list)
    .tolist()
)

X_list, y_list = [], []
for s in sessions:
    for t in range(1, len(s)):
        X_list.append(s[:t])
        y_list.append(s[t])

MAX_LEN = 8
X_pad = pad_sequences(X_list, maxlen=MAX_LEN, padding="pre")
y_arr = np.array(y_list)
print(f"training pairs: {len(X_pad)}")


training pairs: 5806


In [10]:
# 3 layers: embed -> lstm -> softmax over vocab
model = Sequential([
    Embedding(vocab_size, 32, mask_zero=True),
    LSTM(64),
    Dense(vocab_size, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X_pad, y_arr, epochs=15, batch_size=64, validation_split=0.2)


Epoch 1/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.0504 - loss: 4.8790 - val_accuracy: 0.0542 - val_loss: 4.2182
Epoch 2/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0465 - loss: 4.1405 - val_accuracy: 0.0620 - val_loss: 3.9731
Epoch 3/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0839 - loss: 3.8973 - val_accuracy: 0.1678 - val_loss: 3.6804
Epoch 4/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1691 - loss: 3.5688 - val_accuracy: 0.2453 - val_loss: 3.3286
Epoch 5/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2616 - loss: 3.1977 - val_accuracy: 0.3098 - val_loss: 2.9963
Epoch 6/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3206 - loss: 2.9005 - val_accuracy: 0.3683 - val_loss: 2.7184
Epoch 7/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3448 - loss: 2.6328 - val_accuracy: 0.3657 - val_loss: 2.5281
Epoch 8/15
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3506 - loss: 2.4674 - val_accuracy: 0.3761 - val_loss

In [11]:
def predict_next(calls, top=3):
    ids = le.transform(calls)
    ids_pad = pad_sequences([ids], maxlen=MAX_LEN, padding="pre")
    probs = model.predict(ids_pad, verbose=0)[0]
    top_idx = probs.argsort()[-top:][::-1]
    return [(le.inverse_transform([i])[0], round(float(probs[i]), 3)) for i in top_idx]

print("after /products?id=44:")
for call, p in predict_next(["GET /products?id=44"]):
    print(f"  {p*100:5.1f}%  {call}")

print("\nafter applying a female + skirt filter:")
filter_call = 'POST /products/filter  body={"gender":"female","category":"skirt"}'
for call, p in predict_next(["GET /products/filter", filter_call]):
    print(f"  {p*100:5.1f}%  {call}")

print("\nafter a full session ending in /cart:")
for call, p in predict_next(["GET /products", "GET /products?id=44", "GET /cart?product_id=44"]):
    print(f"  {p*100:5.1f}%  {call}")


after /products?id=44:
   48.1%  GET /cart?product_id=44
   13.7%  GET /reviews?product_id=44
    2.8%  GET /cart?product_id=39

after applying a female + skirt filter:
    8.1%  GET /products?id=105
    8.0%  GET /products?id=88
    7.0%  GET /products?id=33

after a full session ending in /cart:
   52.8%  GET /checkout
   40.5%  GET /home
    1.1%  POST /payment  body={"amount": 49.99, "method": "card"}


In [12]:
model.save("cache_predict_model.keras")
print("saved cache_predict_model.keras")


saved cache_predict_model.keras
